In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
#Eu deveira ter importado tudo de uma só vez, porém o código ficou lento pra rodar aqui.
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score


**Treinamento do modelo, adicionando os arquivos**

In [ ]:
# Carregando os dados
train_data = pd.read_csv('/kaggle/input/playground-series-s5e4/train.csv')

test_data = pd.read_csv('/kaggle/input/playground-series-s5e4/test.csv')

submission = pd.read_csv('/kaggle/input/playground-series-s5e4/sample_submission.csv')



**Achando os tipos e as dimensões dos dados**

In [ ]:
print ("train_data: ",train_data.shape)
print ("test_data: ",test_data.shape)
print ("submission: ",submission.shape)

**Lendo e entendendo as colunas dos arquivos, tamanho e algumas infos**

In [ ]:

print ("train_data columns: ",train_data.columns)
print ("test_data columns: ",test_data.columns)
print ("submission colunas: ",submission.columns)

In [ ]:
print ("train_data size: ", train_data.size)
print ("test_data size: ", test_data.size)
print ("submission size: ", submission.size)

In [ ]:
print ("train_data info: ",train_data.info)
print ("test_data info: ",test_data.info)
print ("submission info: ",submission.info)

In [ ]:
train_data.describe()


In [ ]:
test_data.describe()

In [ ]:
#verificando se há valores vazios nos dataset de treino
train_data.isna().sum()

In [ ]:
# Valores ausentes
print(train_data.isnull().sum())

#que legal, também funciona.

**Achando o tempo de escuta minuto/frequencia via sns**

In [ ]:
# Distribuição da variável alvo

plt.figure(figsize=(10, 6))
sns.histplot(train_data['Listening_Time_minutes'], bins=30, kde=True)
plt.title('Distribuição do Tempo de Escuta')
plt.xlabel('Tempo de Escuta (minutos)')
plt.ylabel('Frequência')
plt.show()

**Procurar correlações via mapa de calor**

obs: O método Pandas dataframe.corr() é usado para encontrar a correlação em pares de todas as colunas do Pandas DataFrame em Python. Quaisquer valores NaN são excluídos automaticamente. Para ignorar quaisquer valores não numéricos, use o parâmetro numeric_only = True. Neste artigo, aprenderemos sobre o método DataFrame.corr() em Python .

In [ ]:
# Correlação entre variáveis
plt.figure(figsize=(12, 8))

#correlation_matrix = train_data.corr()
#mapa de calor
sns.heatmap(train_data.corr(), annot=True, fmt=".2f", cmap='coolwarm', numeric_only = False)
plt.title('Matriz de Correlação')
plt.show()

**procurando a xbox, do tempo de escuta por gênero**

In [ ]:
# Análise da variável alvo em relação a outras variáveis
plt.figure(figsize=(12, 6))
sns.boxplot(x='Genre', y='Listening_Time_minutes', data=train_data)
plt.title('Tempo de Escuta por Gênero')
plt.xticks(rotation=45)
plt.show()

In [ ]:
# Análise da variável alvo em relação a Host_Popularity_percentage
plt.figure(figsize=(12, 6))
sns.boxplot(x='Host_Popularity_percentage', y='Listening_Time_minutes', data=train_data)
plt.title('Tempo de Escuta por Popularidade do host')
plt.xticks(rotation=45)
plt.show()

In [ ]:
plt.figure(figsize=(15, 8))
sns.histplot(train_data['Listening_Time_minutes'], bins=30, kde=True)
plt.title('Distribuição do Tempo de Escuta')
plt.xlabel('Host_Popularity_percentage')
plt.ylabel('Listening_Time_minutes')
plt.show()

In [ ]:
#Analise do tempo de escuta com relação a Number_of_Ads

plt.figure (figsize=(12,6))
sns.boxplot (x='Number_of_Ads', y='Listening_Time_minutes', data=train_data)
plt.title('Tempo de escuta x numero de ads')
plt.xticks(rotation=45)
plt.show()


**Tratamento de Valores Ausentes, cuidado**

In [ ]:
# Tratamento de valores ausentes
# Exemplo: preencher com a média ou mediana, ou remover linhas/colunas
# Atenção e cuidado
train_data.fillna(train_data.mean(), inplace=True)  # Para colunas numéricas
train_data.dropna(subset=['Podcast_Name', 'Episode_Title'], inplace=True)  # Para colunas categóricas

**Procurando os Outliers**

In [ ]:
# Detecção de outliers usando o método IQR
Q1 = train_data['Listening_Time_minutes'].quantile(0.25)
Q3 = train_data['Listening_Time_minutes'].quantile(0.75)
IQR = Q3 - Q1

# Filtrando outliers
train_data = train_data[~((train_data['Listening_Time_minutes'] < (Q1 - 1.5 * IQR)) | (train_data['Listening_Time_minutes'] > (Q3 + 1.5 * IQR)))]

In [ ]:
# Codificação de variáveis categóricas
train_data = pd.get_dummies(train_data, columns=['Podcast_Name', 'Genre', 'Episode_Title'], drop_first=True)

**Normalização**

In [ ]:


# Normalização/Padronização
scaler = StandardScaler()
train_data[['Episode_Length_minutes', 'Host_Popularity_percentage', 'Guest_Popularity_percentage', 'Number_of_Ads']] = scaler.fit_transform(train_df[['Episode_Length_minutes', 'Host_Popularity_percentage', 'Guest_Popularity_percentage', 'Number_of_Ads']])

**Divisão entre Dados de Treino, Validação e Teste**

In [ ]:


# Separar variáveis independentes e dependentes
X = train_data.drop(columns=['Listening_Time_minutes', 'id'])
y = train_data['Listening_Time_minutes']

# Divisão em treino e validação
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

**Teste floresta aleatoria**

In [ ]:


# Divisão dos dados em conjunto de treino e teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Treinamento do modelo Random Forest
train_data = RandomForestRegressor(random_state=42)
train_data.fit(X_train, y_train)

# Previsões e cálculo do RMSE
y_pred_baseline = baseline_model.predict(X_test)
rmse_baseline = np.sqrt(mean_squared_error(y_test, y_pred_baseline))
print(f'RMSE do modelo baseline (Random Forest): {rmse_baseline}')

**Hiperparametros**

In [ ]:


# Definindo os parâmetros para o Grid Search
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# Configurando o Grid Search
grid_search = GridSearchCV(RandomForestRegressor(random_state=42), param_grid, cv=5, scoring='neg_mean_squared_error')
grid_search.fit(X_train, y_train)

# Melhor combinação de hiperparâmetros
best_params = grid_search.best_params_
print(f'Melhores hiperparâmetros para Random Forest: {best_params}')

**Novos modelos**

In [ ]:
# Treinando o modelo com os melhores hiperparâmetros
best_rf_model = RandomForestRegressor(**best_params, random_state=42)
best_rf_model.fit(X_train, y_train)

# Previsões e cálculo do RMSE
y_pred_best_rf = best_rf_model.predict(X_test)
rmse_best_rf = np.sqrt(mean_squared_error(y_test, y_pred_best_rf))
print(f'RMSE do modelo Random Forest com melhores hiperparâmetros: {rmse_best_rf}')

**Exemplo validação cruzada**

In [ ]:


# Validação cruzada com K-Fold
cv_scores = cross_val_score(best_rf_model, X, y, cv=5, scoring='neg_mean_squared_error')
cv_rmse = np.sqrt(-cv_scores)
print(f'RMSE médio da validação cruzada para Random Forest: {cv_rmse.mean()}')


**Submissão**

In [1]:
'''submission = train_data.train({
    'id': test_data['id'],
    'Listening_Time_minutes': y_test_data_pred
})

sample_submission.to_csv('submission.csv', index=False)

print("Submissão gerada com sucesso!")'''

''''''
# Submission
# -------------------------------------------
#submission = pd.read_csv('/kaggle/input/playground-series-s5e4/sample_submission.csv')
#submission['Listening_Time_minutes'] = ensemble_test_preds

submission = train_data.train({'id': test_data['id'], 'Listening_Time_minutes': y_test_data_pred})

submission.to_csv('/kaggle/input/playground-series-s5e4/sample_submission.csv', index=False)
print("Submissao creada!")


NameError: name 'ensemble_test_preds' is not defined